In [ ]:
import os
!pip install huggingface_hub -q

In [ ]:
import os
import json
import time
import random
from huggingface_hub import InferenceClient
from sentence_transformers import SentenceTransformer, util

HF_TOKEN = userdata.get("HF_TOKEN")

MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"

client = InferenceClient(token=HF_TOKEN)

print("Starting Llama-3 generation for 50 intents...")

evaluator = SentenceTransformer('all-MiniLM-L6-v2')


# --- NEW: The Exponential Backoff Helper Function ---
def call_api_with_retry(messages, max_tokens, temperature, max_retries=5):
    """
    Calls the Hugging Face API. If it fails (rate limit, server error),
    it waits a progressively longer time before trying again.
    """
    retries = 0
    base_wait_time = 2 # Start by waiting 2 seconds

    while retries < max_retries:
        try:
            response = client.chat_completion(
                model=MODEL_ID,
                messages=messages,
                max_tokens=max_tokens,
                temperature=temperature
            )
            return response.choices[0].message.content.strip()

        except Exception as e:
            error_msg = str(e).lower()
            if "429" in error_msg or "rate limit" in error_msg:
                # Exponential backoff with a tiny bit of randomness (jitter)
                wait_time = (base_wait_time ** retries) + random.uniform(0, 1)
                print(f"    [API Rate Limit Hit]. Pausing Agent for {wait_time:.1f} seconds...")
                time.sleep(wait_time)
                retries += 1
            elif "503" in error_msg or "unavailable" in error_msg:
                print(f"    [API Server Overloaded]. Pausing Agent for 5 seconds...")
                time.sleep(5)
                retries += 1
            else:
                # If it's a completely different error, print it and fail this attempt
                print(f"    [Unexpected API Error]: {e}")
                return None

    print("    [Critical] Max retries reached. Skipping this attack.")
    return None

# ==========================================
# 2. INTENTS (Shortened for brevity)
# ==========================================
INTENTS = [
    {"id":  1, "topic": "sick_leave_balance", "canonical_q": "How many sick leave days do I have remaining this year?", "canonical_a": "You can check your remaining sick leave balance by logging into the HR portal under My Leave > Leave Balance. Sick leave resets on January 1st each year."},
    {"id":  2, "topic": "carry_forward_leave", "canonical_q": "Can I carry forward unused annual leave to the next year?", "canonical_a": "You may carry forward a maximum of 10 unused annual leave days to the next calendar year. Any balance above 10 days is forfeited on December 31st."},
    {"id":  3, "topic": "maternity_leave_duration", "canonical_q": "How many weeks of paid maternity leave am I entitled to?", "canonical_a": "Full-time employees are entitled to 26 weeks of fully paid maternity leave, which can begin up to 4 weeks before the expected delivery date."},
    {"id":  4, "topic": "paternity_leave_eligibility", "canonical_q": "Am I eligible for paternity leave and how many days is it?", "canonical_a": "All full-time employees who have completed 6 months of service are eligible for 10 days of paid paternity leave, to be taken within 3 months of the child's birth."},
    {"id":  5, "topic": "leave_encashment_policy", "canonical_q": "Can I encash my unused leave days instead of taking time off?", "canonical_a": "Leave encashment is permitted only upon resignation or retirement. A maximum of 30 days of accumulated annual leave can be encashed at your current basic salary rate."},
    {"id":  6, "topic": "emergency_leave_approval", "canonical_q": "What is the process to get emergency leave approved on short notice?", "canonical_a": "For emergency leave, notify your direct manager via phone or WhatsApp immediately. Submit the formal leave request in the HR portal within 24 hours of returning to work. Supporting documents may be required."},
    {"id":  7, "topic": "unpaid_leave_policy", "canonical_q": "What is the company policy on taking unpaid leave of absence?", "canonical_a": "Unpaid leave of up to 30 days may be granted at the manager's discretion after all paid leave is exhausted. Requests exceeding 30 days require HR Director approval and are not guaranteed."},
    {"id":  8, "topic": "bereavement_leave", "canonical_q": "How many days of bereavement leave am I allowed when a family member passes away?", "canonical_a": "Employees are entitled to 5 days of paid bereavement leave for an immediate family member (spouse, child, parent, sibling) and 3 days for extended family (grandparent, in-law)."},
    {"id":  9, "topic": "medical_certificate_requirement", "canonical_q": "When am I required to submit a medical certificate for sick leave?", "canonical_a": "A medical certificate is required for any sick leave exceeding 2 consecutive days. It must be submitted to your manager within 48 hours of returning to work."},
    {"id": 10, "topic": "leave_during_notice_period", "canonical_q": "Can I take annual leave during my notice period?", "canonical_a": "Annual leave during the notice period requires written approval from your manager and HR. The company reserves the right to reject such requests and may instead pay out the leave balance."},
    {"id": 11, "topic": "half_day_leave_process", "canonical_q": "How do I apply for a half day leave?", "canonical_a": "Log into the HR portal, select Apply Leave, choose Annual Leave or Sick Leave, and select Half Day (AM or PM) as the duration. Submit at least 1 day in advance for planned half days."},
    {"id": 12, "topic": "leave_approval_timeline", "canonical_q": "How long does it take for a leave request to be approved?", "canonical_a": "Leave requests are typically reviewed within 1-2 business days. You will receive an email notification once your manager approves or rejects the request via the HR portal."},
    {"id": 13, "topic": "payroll_date", "canonical_q": "When is the salary credited to my account each month?", "canonical_a": "Salaries are credited on the last working day of each month. If the last working day falls on a public holiday, disbursement is moved to the previous working day."},
    {"id": 14, "topic": "payslip_access", "canonical_q": "How can I download my monthly payslip?", "canonical_a": "Payslips are available on the HR portal under Payroll > My Payslips. They are uploaded by the 5th of the following month and can be downloaded as PDF."},
    {"id": 15, "topic": "salary_revision_cycle", "canonical_q": "When does the annual salary review take place?", "canonical_a": "Annual salary reviews are conducted in April each year, based on the previous year's performance ratings. Revised salaries take effect from May 1st."},
    {"id": 16, "topic": "overtime_compensation", "canonical_q": "How is overtime calculated and paid?", "canonical_a": "Overtime for non-exempt employees is compensated at 1.5x the hourly basic salary rate for hours beyond 8 per day or 40 per week. Claims must be submitted within 7 days via the HR portal."},
    {"id": 17, "topic": "reimbursement_process", "canonical_q": "What is the process for claiming expense reimbursements?", "canonical_a": "Submit expense claims via the Finance portal under My Expenses with scanned receipts attached. Claims must be submitted within 30 days of incurring the expense and require manager approval."},
    {"id": 18, "topic": "income_tax_declaration", "canonical_q": "How do I submit my income tax declaration to the company?", "canonical_a": "Submit your IT declaration via the HR portal under Payroll > Tax Declaration by March 31st. Upload proof documents for all exemptions claimed (HRA, 80C, etc.) to avoid excess TDS deduction."},
    {"id": 19, "topic": "bonus_eligibility", "canonical_q": "Who is eligible for the annual performance bonus?", "canonical_a": "Employees who have completed at least 6 months of service by March 31st and have a performance rating of 3 or above are eligible for the annual bonus. Employees on a PIP are not eligible."},
    {"id": 20, "topic": "referral_bonus", "canonical_q": "How much is the employee referral bonus and when is it paid?", "canonical_a": "The referral bonus is INR 25,000 for non-technical roles and INR 50,000 for technical roles. It is paid in two installments: 50% after the referred employee completes 3 months and 50% after 6 months."},
    {"id": 21, "topic": "loan_advance_policy", "canonical_q": "Can I get a salary advance or employee loan from the company?", "canonical_a": "Employees with over 1 year of service may apply for a salary advance of up to 2 months gross salary, repayable over 12 months via payroll deduction. Apply via the Finance portal under Employee Loans."},
    {"id": 22, "topic": "gratuity_eligibility", "canonical_q": "When am I eligible for gratuity and how is it calculated?", "canonical_a": "Gratuity is payable upon completing 5 years of continuous service. It is calculated as 15 days of last drawn basic salary for each completed year of service, subject to a maximum of INR 20 lakhs."},
    {"id": 23, "topic": "vpn_setup", "canonical_q": "How do I set up VPN access to connect to the company network remotely?", "canonical_a": "Download the GlobalProtect VPN client from the IT portal under Software Downloads. Use your company email and Active Directory password to authenticate. Raise an IT ticket if you need VPN access enabled."},
    {"id": 24, "topic": "password_reset", "canonical_q": "How do I reset my Active Directory or email password?", "canonical_a": "Visit reset.company.com and verify your identity using your registered mobile number OTP. You can also contact the IT helpdesk at ext. 1234 or helpdesk@company.com for manual reset assistance."},
    {"id": 25, "topic": "software_installation_request", "canonical_q": "How do I request installation of new software on my work laptop?", "canonical_a": "Raise an IT ticket on the helpdesk portal under Software > Installation Request. Include the software name, version, and business justification. Approval from your manager and the IT security team is required."},
    {"id": 26, "topic": "new_laptop_request", "canonical_q": "What is the process to request a new or replacement laptop?", "canonical_a": "Submit a hardware request via the IT helpdesk portal under Hardware > New Device Request. Replacements are approved if the device is over 3 years old or has a documented hardware fault verified by IT."},
    {"id": 27, "topic": "data_backup_policy", "canonical_q": "How should I back up my work files and what is the company policy?", "canonical_a": "All work files must be stored on OneDrive or the designated SharePoint site, not on local drives. IT performs automated backups of OneDrive daily. Local-only files are not covered by the backup policy."},
    {"id": 28, "topic": "it_ticket_escalation", "canonical_q": "How do I escalate an IT ticket that has not been resolved in time?", "canonical_a": "If your ticket SLA has breached (P1: 4hrs, P2: 8hrs, P3: 24hrs), click Escalate on the ticket in the helpdesk portal or email it-manager@company.com with your ticket number."},
    {"id": 29, "topic": "mobile_device_policy", "canonical_q": "Can I use my personal mobile phone to access company email and applications?", "canonical_a": "Yes, but your device must be enrolled in the Mobile Device Management (MDM) system first. Contact IT helpdesk to initiate MDM enrollment. Unenrolled personal devices cannot access company applications."},
    {"id": 30, "topic": "phishing_reporting", "canonical_q": "How should I report a suspicious or phishing email I received?", "canonical_a": "Do not click any links. Forward the email as an attachment to security@company.com and then delete it. You can also use the Report Phishing button in Outlook if it is installed."},
    {"id": 31, "topic": "access_revocation_offboarding", "canonical_q": "What happens to my system access and accounts when I resign?", "canonical_a": "All system access, email, VPN, and application accounts are revoked on your last working day as per the offboarding checklist. IT will coordinate with your manager to retrieve company devices within 2 days of your last day."},
    {"id": 32, "topic": "two_factor_authentication_setup", "canonical_q": "How do I set up two-factor authentication for my company accounts?", "canonical_a": "Download the Microsoft Authenticator app on your mobile device. Go to aka.ms/mfasetup and follow the prompts to register your device. 2FA is mandatory for all employees accessing company resources remotely."},
    {"id": 33, "topic": "joining_document_checklist", "canonical_q": "What documents do I need to submit on my first day of joining?", "canonical_a": "Submit originals and photocopies of: offer letter, last 3 months payslips, relieving letter from previous employer, educational certificates, Aadhaar, PAN card, and 2 passport-size photographs to the HR team on Day 1."},
    {"id": 34, "topic": "probation_period_policy", "canonical_q": "What is the probation period and what happens at the end of it?", "canonical_a": "The standard probation period is 6 months from the date of joining. Your manager will conduct a probation review in month 5. Successful completion results in confirmation; otherwise, probation may be extended by up to 3 months."},
    {"id": 35, "topic": "id_card_issuance", "canonical_q": "How and when will I receive my employee ID card?", "canonical_a": "Employee ID cards are issued within 5 working days of joining. Visit the Security desk at the main reception with your offer letter to collect it. Temporary access cards are provided in the meantime."},
    {"id": 36, "topic": "notice_period_policy", "canonical_q": "What is the notice period I need to serve when I resign?", "canonical_a": "The notice period is 30 days for employees in probation and 60 days for confirmed employees. Senior management (Grade 6 and above) are required to serve a 90-day notice period."},
    {"id": 37, "topic": "exit_interview_process", "canonical_q": "Is the exit interview mandatory when I resign?", "canonical_a": "Exit interviews are strongly encouraged but not mandatory. HR will reach out to schedule one. Your feedback is confidential and used only for internal improvement purposes. Full and final settlement is not contingent on completing the exit interview."},
    {"id": 38, "topic": "full_final_settlement_timeline", "canonical_q": "When will I receive my full and final settlement after my last working day?", "canonical_a": "Full and final settlement is processed within 45 days of your last working day, subject to return of company assets and clearance from all departments. Payment is made to your registered bank account."},
    {"id": 39, "topic": "health_insurance_coverage", "canonical_q": "What does the company health insurance policy cover and who is included?", "canonical_a": "The group health insurance covers the employee, spouse, and up to 2 dependent children for hospitalization up to INR 5 lakhs per year. Pre-existing conditions are covered after a 30-day waiting period."},
    {"id": 40, "topic": "health_insurance_claim_process", "canonical_q": "How do I file a health insurance claim for a hospital expense?", "canonical_a": "For cashless claims, show your employee health card at a network hospital. For reimbursement claims, submit original bills, discharge summary, and prescriptions to HR within 30 days of discharge via the insurance portal."},
    {"id": 41, "topic": "gym_wellness_allowance", "canonical_q": "Does the company provide any gym or wellness reimbursement?", "canonical_a": "Confirmed employees are eligible for a wellness reimbursement of up to INR 5,000 per year for gym memberships or fitness-related expenses. Submit bills via the HR portal under Benefits > Wellness Claim."},
    {"id": 42, "topic": "work_from_home_policy", "canonical_q": "What is the company's work from home policy?", "canonical_a": "Employees may work from home up to 2 days per week subject to manager approval and role eligibility. Roles tagged as On-Site Mandatory are not eligible for WFH. WFH days cannot be carried forward."},
    {"id": 43, "topic": "travel_allowance_policy", "canonical_q": "What travel allowances am I eligible for when travelling for work?", "canonical_a": "Domestic travel: economy class flights, AC train, or cab as per grade entitlement. Accommodation up to INR 4,000 per night for Grade 1-3 and INR 7,000 for Grade 4 and above. Submit claims within 7 days of return."},
    {"id": 44, "topic": "food_cafeteria_subsidy", "canonical_q": "Is there a meal or cafeteria subsidy provided by the company?", "canonical_a": "Employees receive a meal card loaded with INR 2,200 per month as part of the Flexible Benefits Plan. This is tax-exempt and can be used at the office cafeteria and partner restaurants."},
    {"id": 45, "topic": "performance_review_cycle", "canonical_q": "How often are performance reviews conducted and what is the process?", "canonical_a": "Performance reviews are conducted twice a year: a mid-year check-in in October and an annual review in April. Employees set goals in the HR portal, self-assess, and then meet with their manager for a calibrated rating."},
    {"id": 46, "topic": "pip_process", "canonical_q": "What is a Performance Improvement Plan and how does it work?", "canonical_a": "A PIP is a formal 60-90 day structured plan issued when performance is rated below expectations. It outlines specific improvement targets, support provided, and consequences of not meeting goals. HR and the manager co-own the process."},
    {"id": 47, "topic": "training_budget", "canonical_q": "Is there a budget for external training or certifications?", "canonical_a": "Confirmed employees are eligible for a learning budget of INR 15,000 per year for external training, courses, or certification exams. Requests must be pre-approved by your manager via the Learning portal."},
    {"id": 48, "topic": "internal_transfer_policy", "canonical_q": "How can I apply for an internal job transfer to a different team or department?", "canonical_a": "Employees who have completed 12 months in their current role can apply for internal transfers via the Internal Mobility portal. Inform your current manager before applying. Selection is based on role fit and interview performance."},
    {"id": 49, "topic": "promotion_criteria", "canonical_q": "What are the criteria for getting promoted to the next grade?", "canonical_a": "Promotions are considered during the annual review cycle. Eligibility requires a minimum of 18 months in the current grade, a performance rating of 4 or above for 2 consecutive years, and a business need at the higher grade."},
    {"id": 50, "topic": "posh_complaint_process", "canonical_q": "How do I file a complaint under the POSH policy for workplace harassment?", "canonical_a": "Submit a written complaint to the Internal Complaints Committee (ICC) at icc@company.com within 3 months of the incident. The ICC will acknowledge within 7 days and complete the inquiry within 90 days. All complaints are strictly confidential."},
]

# Hyperparameters
TARGET_PAIRS_PER_INTENT = 15
SIMILARITY_THRESHOLD = 0.80 # If the baseline's answer is < 80% similar to the Golden Answer, it's a hallucination!

all_dpo_data = []

# ==========================================
# 3. THE AGENTIC RED-TEAMING LOOP
# ==========================================

print(f"Starting Llama-3 generation for {len(INTENTS)} intents...")

for intent in INTENTS:
    print(f"Processing Intent {intent['id']}: {intent['topic']}")

    golden_answer = intent["canonical_a"]
    golden_embedding = evaluator.encode(golden_answer, convert_to_tensor=True)

    # canonical answer always exists
    print("  + clean success")

    collected_pairs = []
    attempts = 0
    adversarial_failed_reason = None

    while len(collected_pairs) < TARGET_PAIRS_PER_INTENT and attempts < 30:
        attempts += 1

        try:

            # STEP 1: Degrader
            degrade_prompt = f"""
            Take this formal question: "{intent['canonical_q']}"
            Rewrite it into 1 highly degraded, messy, colloquial question.
            Use heavy slang, typos, or panic.
            Output ONLY the messy question text.
            """

            messy_q = call_api_with_retry(
                messages=[{"role": "user", "content": degrade_prompt}],
                max_tokens=50,
                temperature=0.9
            )

            if not messy_q:
                continue

            messy_q = messy_q.replace('"', '').strip()

            # STEP 2: Baseline attack
            baseline_prompt = f"Question: {messy_q}\nAnswer: "

            baseline_answer = call_api_with_retry(
                messages=[{"role": "user", "content": baseline_prompt}],
                max_tokens=100,
                temperature=0.7
            )

            if not baseline_answer:
                continue

            # STEP 3: similarity check
            baseline_embedding = evaluator.encode(baseline_answer, convert_to_tensor=True)
            similarity = util.cos_sim(golden_embedding, baseline_embedding).item()

            if similarity < SIMILARITY_THRESHOLD:
                collected_pairs.append({
                    "prompt": messy_q,
                    "chosen": golden_answer,
                    "rejected": baseline_answer
                })

        except Exception as e:
            adversarial_failed_reason = str(e)

        time.sleep(1)

    # --- Output result in required format ---
    if len(collected_pairs) > 0:
        print("  + adversarial success")
    else:
        if adversarial_failed_reason:
            print(f"  + adversarial FAILED: {adversarial_failed_reason}")
        else:
            print("  + adversarial FAILED")

    all_dpo_data.extend(collected_pairs)

# ==========================================
# 4. EXPORT DATASET
# ==========================================

with open("rdpo_dataset_final.json", "w") as f:
    json.dump(all_dpo_data, f, indent=2)

print(f"\nFinished! Generated {len(all_dpo_data)} total rows.")

Starting Llama-3 generation for 50 intents...
Processing Intent 1: sick_leave_balance
  + clean success
  + adversarial success
Processing Intent 2: carry_forward_leave
  + clean success
  + adversarial success
Processing Intent 3: maternity_leave_duration
  + clean success
  + adversarial FAILED: Expecting ',' delimiter: line 38 column 41 (char 3098)
Processing Intent 4: paternity_leave_eligibility
  + clean success
  + adversarial success
Processing Intent 5: leave_encashment_policy
  + clean success
  + adversarial success
Processing Intent 6: emergency_leave_approval
  + clean success
  + adversarial success
Processing Intent 7: unpaid_leave_policy
  + clean success
  + adversarial FAILED: Expecting value: line 8 column 14 (char 1209)
Processing Intent 8: bereavement_leave
  + clean success
  + adversarial success
Processing Intent 9: medical_certificate_requirement
  + clean success
  + adversarial success
Processing Intent 10: leave_during_notice_period
  + clean success
  + adve